In [0]:
%pip install beautifulsoup4

try:    

    from bs4 import BeautifulSoup
    print("✅ BeautifulSoup4 is installed and working correctly.")
except ImportError:    

    print("❌ Error: BeautifulSoup4 failed to install. Please re-run this cell.")

In [0]:
# import requests
# from bs4 import BeautifulSoup
# import os
# from datetime import datetime, timedelta

# # -------------------------------------------------------------------------
# # 1. Azure Storage Config (PASTE YOUR INFO BELOW)
# # -------------------------------------------------------------------------
# storage_account_name = "REMOVED_FOR_SECURITY"  # <-- PASTE Name here
# storage_account_key = "REMOVED_FOR_SECURITY"   # <-- PASTE Key here

# # Connect to ADLS
# spark.conf.set(
#     f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
#     storage_account_key
# )

# # Auto-create folders if missing
# spark.conf.set("fs.azure.createRemoteFileSystemDuringInitialization", "true")

# # -------------------------------------------------------------------------
# # 2. The Reusable Ingestion Function
# # -------------------------------------------------------------------------
# def ingest_dataset(dataset_name, page_url, file_pattern):
#     print(f"\n🚀 STARTING INGESTION: {dataset_name.upper()}")

#     # Define Dynamic Paths
#     base_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/{dataset_name}"
#     landing_zone = f"{base_path}/landing/"
#     archive_zone = f"{base_path}/archive/"

#     # A. Link Finder
#     print(f"  🔍 Finding download link on page...")
#     headers = {'User-Agent': 'Mozilla/5.0'}
#     response = requests.get(page_url, headers=headers)
#     soup = BeautifulSoup(response.content, 'html.parser')

#     fresh_url = None
#     for link in soup.find_all('a', href=True):
#         if "download" in link['href'] and file_pattern in link['href']:
#             fresh_url = f"https://www.dubaipulse.gov.ae{link['href']}" if link['href'].startswith("/") else link['href']
#             print(f"  ✅ Found link: {fresh_url}")
#             break

#     if not fresh_url: raise Exception(f"❌ Link not found for {file_pattern}")

#     # B. Archive Current File (if exists)
#     try:
#         dbutils.fs.ls(f"{landing_zone}current.csv")
#         today = datetime.now().strftime("%Y-%m-%d")
#         dbutils.fs.mv(f"{landing_zone}current.csv", f"{archive_zone}{today}.csv")
#         print(f"  📦 Archived old file.")
#     except: pass

#     # C. Download & Upload
#     local_path = f"/tmp/{dataset_name}.csv"
#     print(f"  ⬇️ Downloading & Uploading...")
#     with requests.get(fresh_url, headers=headers, stream=True) as r:
#         r.raise_for_status()
#         with open(local_path, 'wb') as f:
#             for chunk in r.iter_content(chunk_size=10*1024*1024):
#                 f.write(chunk)

#     dbutils.fs.cp(f"file:{local_path}", f"{landing_zone}current.csv")
#     os.remove(local_path)
#     print(f"✅ FINISHED: {dataset_name} is ready.")

# print("Master Function Loaded.")
# CELL 2: Setup & Config
import os
from datetime import datetime, timedelta

# Azure Storage Config
storage_account_name = "REMOVED_FOR_SECURITY"
storage_account_key = "REMOVED_FOR_SECURITY"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)
spark.conf.set("fs.azure.createRemoteFileSystemDuringInitialization", "true")

print("✅ Configuration Loaded.")

In [0]:
# # CELL 3: Execute Ingestion
# # -------------------------------------------------------------------------

# # 1. Ingest Units (Product Dimension)
# ingest_dataset(
#     dataset_name="units",
#     page_url="https://www.dubaipulse.gov.ae/data/dld-registration/dld_units-open",
#     file_pattern="units.csv"
# )

# # 2. Ingest Projects (Status Dimension)
# ingest_dataset(
#     dataset_name="projects",
#     page_url="https://www.dubaipulse.gov.ae/data/dld-registration/dld_projects-open",
#     file_pattern="projects.csv"
# )
# CELL 3: Verify Dimension Files Exist (Skipping scraping - site is down)
print("🔍 Verifying dimension files in Bronze layer...")

storage_account_name = "adlsdubaisameer2026"

files_to_check = {
    "units": f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/units/landing/current.csv",
    "projects": f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/projects/landing/current.csv"
}

all_good = True
for dataset, file_path in files_to_check.items():
    try:
        dbutils.fs.ls(file_path)
        print(f"✅ Found: {dataset}/landing/current.csv")
    except Exception:
        print(f"❌ MISSING: {dataset}/landing/current.csv")
        all_good = False

if all_good:
    print("\n🚀 All dimension files verified! Pipeline can proceed.")
else:
    print("\n⚠️ Some files are missing. Please upload them manually to Azure.")